In [1]:
import os

root_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
print(f"Root directory: {root_dir}")
files_path = os.path.join(root_dir, "data", "input", "raw", "files")

Root directory: /Users/vicvictor/programming/curso-ia


In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client["soberania_ufpi"]
collection = db["departamento_computacao"]

print(f"Conectado ao banco: {db.name}")
print(f"Collection: {collection.name}")
print(f"Total de documentos: {collection.count_documents({})}")

Conectado ao banco: soberania_ufpi
Collection: departamento_computacao
Total de documentos: 1


In [17]:
import shutil

# Quantidade de arquivos a processar (None = todos)
N = None

files_to_rename_path = os.path.join(root_dir, "data", "input", "raw", "files_to_rename")
os.makedirs(files_to_rename_path, exist_ok=True)

# ------------------------------------------------------------
processed = 0
copied = 0
not_found = 0
errors = []

for doc in collection.find():
    for turma in doc.get("turmas", []):
        for plano in turma.get("planos_aula", []):
            for arquivo in plano.get("arquivos", []):
                path_arquivo = arquivo.get("path_arquivo")
                nome_arquivo = arquivo.get("nome_arquivo")

                if not path_arquivo or not nome_arquivo:
                    continue

                src = os.path.join(files_path, path_arquivo.replace("/", os.sep))
                subdir = os.path.dirname(path_arquivo.replace("/", os.sep))
                dst_dir = os.path.join(files_to_rename_path, subdir)
                os.makedirs(dst_dir, exist_ok=True)
                dst = os.path.join(dst_dir, nome_arquivo)

                processed += 1

                if not os.path.exists(src):
                    not_found += 1
                    errors.append(f"[NÃO ENCONTRADO] {src}")
                    print(f"  [NÃO ENCONTRADO] {src}")
                elif os.path.exists(dst):
                    print(f"  [JÁ EXISTE]      {dst}")
                else:
                    shutil.copy2(src, dst)
                    copied += 1
                    print(f"  [COPIADO]        {os.path.basename(src)} -> {dst}")

                if N is not None and processed >= N:
                    break
            if N is not None and processed >= N:
                break
        if N is not None and processed >= N:
            break
    if N is not None and processed >= N:
        break

print("\n--- Resumo ---")
print(f"Processados : {processed}")
print(f"Copiados    : {copied}")
print(f"Não achados : {not_found}")

with open(os.path.join(files_to_rename_path, "erros.txt"), "w") as f:
    for error in errors:
        f.write(error + "\n")

  [JÁ EXISTE]      /Users/vicvictor/programming/curso-ia/data/input/raw/files_to_rename/2026/02/20/EstruturasDeDados-60h.pdf
  [JÁ EXISTE]      /Users/vicvictor/programming/curso-ia/data/input/raw/files_to_rename/2025/08/04/EstruturasDeDados-60h.pdf
  [JÁ EXISTE]      /Users/vicvictor/programming/curso-ia/data/input/raw/files_to_rename/2025/08/14/aula01_introducaoPythonBasico.pdf
  [JÁ EXISTE]      /Users/vicvictor/programming/curso-ia/data/input/raw/files_to_rename/2025/08/21/termosFrequentes.py
  [JÁ EXISTE]      /Users/vicvictor/programming/curso-ia/data/input/raw/files_to_rename/2025/08/26/aula_Array_ListaEncadeada.pdf
  [COPIADO]        6888487 -> /Users/vicvictor/programming/curso-ia/data/input/raw/files_to_rename/2025/08/26/Lista.py
  [COPIADO]        6888491 -> /Users/vicvictor/programming/curso-ia/data/input/raw/files_to_rename/2025/08/26/AppLista.py
  [COPIADO]        6894504 -> /Users/vicvictor/programming/curso-ia/data/input/raw/files_to_rename/2025/08/28/aula_Pilha_Stack.p